# YouTube Data Collection — Nursing

Collects YouTube **comments and video transcripts** about nursing in Australia for sentiment analysis and topic modelling.

**Scope (Week 2 deliverable):**
- Searches YouTube for nursing-related videos (Australian context)
- ~20 videos, top-level comments + transcripts where available
- Captures text, publish date, likes, video metadata
- Saves to `data/raw/` as JSON and CSV

**Two data sources, one schema.** Comments and transcripts are saved into separate files but with a shared `source` column so they can be combined for sentiment/topic modelling. Keep them tagged — a transcript is a creator's monologue (often promotional), a comment is a viewer's reaction. Mixing them blindly in sentiment averages will produce misleading trends.

**Designed to expand:** the `OCCUPATIONS` dict can be extended without changing the rest of the notebook.

**API key handling:** loads from `.env` file via `python-dotenv`. Never commit `.env`.

## 1. Setup

### One-time installs

In [ ]:
%pip install --quiet google-api-python-client youtube-transcript-api pandas python-dotenv

### `.env` file setup

Create a file called `.env` in the project root (same folder as `package.json`) with this line:

```
YOUTUBE_API_KEY=your_key_here
```

**Add `.env` to `.gitignore`.** Never commit API keys.

If the existing key in the old notebook has been committed to git history, rotate it in Google Cloud Console.

In [ ]:
import os
import json
import time
import logging
from pathlib import Path
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import (
    TranscriptsDisabled,
    NoTranscriptFound,
    VideoUnavailable,
)

# Logging — so errors don't get silently swallowed like in the old notebook
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger(__name__)

load_dotenv()
API_KEY = os.getenv('YOUTUBE_API_KEY')
if not API_KEY:
    raise RuntimeError(
        'YOUTUBE_API_KEY not found. Create a .env file in the project root '
        'with: YOUTUBE_API_KEY=your_key_here'
    )

youtube = build('youtube', 'v3', developerKey=API_KEY)
log.info('YouTube client initialised')

## 2. Configuration

In [ ]:
OCCUPATIONS = {
    'nursing': [
        'nursing Australia',
        'registered nurse Australia',
        'nursing jobs Victoria',
        'nursing career Australia',
        'nurse burnout Australia',
    ],
}

# Collection settings (Light profile)
VIDEOS_PER_OCCUPATION = 20
COMMENTS_PER_VIDEO = 100
REGION_CODE = 'AU'
RELEVANCE_LANGUAGE = 'en'
TRANSCRIPT_LANGUAGES = ['en', 'en-AU', 'en-GB', 'en-US']

OUTPUT_DIR = Path('data/raw')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

log.info(f'Occupations: {list(OCCUPATIONS.keys())}')
log.info(f'Target: {VIDEOS_PER_OCCUPATION} videos × up to {COMMENTS_PER_VIDEO} comments + transcripts')

## 3. Video search

In [ ]:
def search_videos(query, max_results=5, region_code='AU', language='en'):
    """Search YouTube for videos matching query. Returns list of dicts with video metadata."""
    try:
        request = youtube.search().list(
            q=query,
            part='snippet',
            type='video',
            maxResults=max_results,
            regionCode=region_code,
            relevanceLanguage=language,
        )
        response = request.execute()
    except HttpError as e:
        log.error(f'Search failed for "{query}": {e}')
        return []

    videos = []
    for item in response.get('items', []):
        videos.append({
            'video_id': item['id']['videoId'],
            'title': item['snippet']['title'],
            'channel': item['snippet']['channelTitle'],
            'published_at': item['snippet']['publishedAt'],
            'description': item['snippet']['description'],
            'search_query': query,
        })
    return videos


def collect_videos_for_occupation(occupation, search_terms, total_videos):
    """Distribute video budget across search terms. Dedupe by video_id."""
    per_term = max(1, total_videos // len(search_terms))
    seen_ids = set()
    all_videos = []

    for term in search_terms:
        log.info(f'  Searching: "{term}" (target {per_term} videos)')
        videos = search_videos(term, max_results=per_term)
        for v in videos:
            if v['video_id'] not in seen_ids:
                v['occupation'] = occupation
                seen_ids.add(v['video_id'])
                all_videos.append(v)
        time.sleep(0.5)

    log.info(f'  Collected {len(all_videos)} unique videos for {occupation}')
    return all_videos


all_videos = []
for occupation, search_terms in OCCUPATIONS.items():
    log.info(f'Searching for: {occupation}')
    videos = collect_videos_for_occupation(occupation, search_terms, VIDEOS_PER_OCCUPATION)
    all_videos.extend(videos)

log.info(f'TOTAL: {len(all_videos)} videos across all occupations')
pd.DataFrame(all_videos)[['occupation', 'title', 'channel', 'published_at']].head(10)

## 4. Comment collection

Fetches top-level comments per video with pagination, handles disabled comments, logs (not swallows) errors.

In [ ]:
def get_comments_for_video(video_id, max_comments=100):
    """Fetch up to max_comments top-level comments. Returns [] on error (logged)."""
    comments = []
    next_page_token = None

    while len(comments) < max_comments:
        try:
            request = youtube.commentThreads().list(
                part='snippet',
                videoId=video_id,
                maxResults=min(100, max_comments - len(comments)),
                pageToken=next_page_token,
                textFormat='plainText',
            )
            response = request.execute()
        except HttpError as e:
            err_reason = e.error_details[0].get('reason', 'unknown') if e.error_details else 'unknown'
            log.warning(f'  Comments unavailable for {video_id}: {err_reason}')
            return comments

        for item in response.get('items', []):
            snippet = item['snippet']['topLevelComment']['snippet']
            comments.append({
                'comment_id': item['id'],
                'video_id': video_id,
                'text': snippet['textDisplay'],
                'author': snippet['authorDisplayName'],
                'published_at': snippet['publishedAt'],
                'updated_at': snippet['updatedAt'],
                'like_count': snippet['likeCount'],
                'reply_count': item['snippet']['totalReplyCount'],
                'source': 'youtube_comment',
            })

        next_page_token = response.get('nextPageToken')
        if not next_page_token:
            break
        time.sleep(0.3)

    return comments


all_comments = []
for i, video in enumerate(all_videos, 1):
    log.info(f'[{i}/{len(all_videos)}] {video["title"][:60]}...')
    comments = get_comments_for_video(video['video_id'], max_comments=COMMENTS_PER_VIDEO)
    for c in comments:
        c['occupation'] = video['occupation']
        c['video_title'] = video['title']
        c['video_channel'] = video['channel']
        c['search_query'] = video['search_query']
    all_comments.extend(comments)
    log.info(f'  → {len(comments)} comments (running total: {len(all_comments)})')

log.info(f'COMMENTS COMPLETE: {len(all_comments)} comments from {len(all_videos)} videos')

## 5. Transcript collection

Fetches video transcripts via `youtube-transcript-api`.

**Important caveats — read before relying on this output:**

1. **IP blocks are common.** YouTube blocks cloud IPs, some university networks, and VPNs. If you get a wall of `IpBlocked` errors, run this from a different network. Errors are logged so you'll see exactly which videos failed and why — no silent failures.
2. **Transcripts are the creator's voice, not viewers'.** Career-advice videos especially contain promotional language. Tag your data with `source` and consider analysing comments and transcripts separately as well as together.
3. **Auto-generated captions are noisy.** Punctuation is approximate, homophones get confused, no speaker labels. This affects topic modelling more than sentiment.
4. **One row per transcript, not per segment.** The API returns time-stamped segments; we concatenate them into a single document. If you want timestamped sentence-level analysis later, modify `get_transcript_for_video` to keep segments.

In [ ]:
def get_transcript_for_video(video_id, languages=None):
    """Fetch transcript for one video. Returns dict or None.
    
    Distinguishes expected failures (no transcript exists) from
    unexpected ones (IP block, network error) — both logged separately.
    """
    languages = languages or ['en']
    try:
        transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=languages)
    except (TranscriptsDisabled, NoTranscriptFound):
        log.info(f'  No transcript available for {video_id}')
        return None
    except VideoUnavailable:
        log.warning(f'  Video unavailable: {video_id}')
        return None
    except Exception as e:
        # Catches IpBlocked, network errors, library internal changes.
        # Not silent — logged so you can diagnose.
        log.error(f'  Transcript fetch failed for {video_id}: {type(e).__name__}: {e}')
        return None

    full_text = ' '.join(segment['text'] for segment in transcript_list)
    return {
        'video_id': video_id,
        'text': full_text,
        'segment_count': len(transcript_list),
        'word_count': len(full_text.split()),
        'source': 'youtube_transcript',
    }


all_transcripts = []
transcript_failures = []

for i, video in enumerate(all_videos, 1):
    log.info(f'[{i}/{len(all_videos)}] Transcript: {video["title"][:60]}...')
    result = get_transcript_for_video(video['video_id'], languages=TRANSCRIPT_LANGUAGES)

    if result is None:
        transcript_failures.append(video['video_id'])
        continue

    # Attach same video-level context as comments → common schema across sources
    result['occupation'] = video['occupation']
    result['video_title'] = video['title']
    result['video_channel'] = video['channel']
    result['published_at'] = video['published_at']
    result['search_query'] = video['search_query']
    all_transcripts.append(result)
    log.info(f'  → {result["word_count"]} words')

    time.sleep(0.5)  # gentle pacing reduces rate-limit risk

log.info(f'TRANSCRIPTS COMPLETE: {len(all_transcripts)} fetched, {len(transcript_failures)} failed')
if transcript_failures and len(transcript_failures) > len(all_videos) * 0.5:
    log.warning(
        f'Over 50% of transcripts failed ({len(transcript_failures)}/{len(all_videos)}). '
        f'This usually means an IP block — try running from a different network.'
    )

## 6. Inspect what we collected

In [ ]:
df_comments = pd.DataFrame(all_comments)
df_transcripts = pd.DataFrame(all_transcripts)

if not df_comments.empty:
    df_comments['published_at'] = pd.to_datetime(df_comments['published_at'])
    print('=== COMMENTS ===')
    print(f'Total: {len(df_comments)}')
    print(f'Unique videos:  {df_comments["video_id"].nunique()}')
    print(f'Date range:     {df_comments["published_at"].min().date()} → {df_comments["published_at"].max().date()}')
    print('Per occupation:')
    print(df_comments.groupby('occupation').size().to_string())
    print('Per year (for ARIMA feasibility):')
    print(df_comments.groupby(df_comments['published_at'].dt.year).size().to_string())

print()
if not df_transcripts.empty:
    df_transcripts['published_at'] = pd.to_datetime(df_transcripts['published_at'])
    print('=== TRANSCRIPTS ===')
    print(f'Total: {len(df_transcripts)} videos with transcripts')
    print(f'Total words: {df_transcripts["word_count"].sum():,}')
    print(f'Median words per transcript: {df_transcripts["word_count"].median():.0f}')
    print('Per occupation:')
    print(df_transcripts.groupby('occupation').size().to_string())
else:
    print('=== TRANSCRIPTS ===')
    print('No transcripts collected. Check logs above — likely an IP block or all videos lack captions.')

In [ ]:
print('Sample comments:')
display(df_comments[['occupation', 'video_title', 'published_at', 'like_count', 'text']].head(3))

if not df_transcripts.empty:
    print('\nSample transcript (first 300 chars):')
    sample = df_transcripts.iloc[0]
    print(f"Video: {sample['video_title']}")
    print(f"Words: {sample['word_count']}")
    print(f"Text: {sample['text'][:300]}...")

## 7. Save to disk

Three output files in `data/raw/`:
- **Comments** — `youtube_comments_nursing_<date>.{json,csv}`
- **Transcripts** — `youtube_transcripts_nursing_<date>.{json,csv}`
- **Video metadata** — `youtube_videos_nursing_<date>.json` (for the bias audit in Week 10)

Comments and transcripts stay separate. The `source` column lets you union them later if needed.

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d')

comments_json = OUTPUT_DIR / f'youtube_comments_nursing_{timestamp}.json'
comments_csv = OUTPUT_DIR / f'youtube_comments_nursing_{timestamp}.csv'
with open(comments_json, 'w', encoding='utf-8') as f:
    json.dump(all_comments, f, ensure_ascii=False, indent=2, default=str)
if not df_comments.empty:
    df_comments.to_csv(comments_csv, index=False, encoding='utf-8')

transcripts_json = OUTPUT_DIR / f'youtube_transcripts_nursing_{timestamp}.json'
transcripts_csv = OUTPUT_DIR / f'youtube_transcripts_nursing_{timestamp}.csv'
with open(transcripts_json, 'w', encoding='utf-8') as f:
    json.dump(all_transcripts, f, ensure_ascii=False, indent=2, default=str)
if not df_transcripts.empty:
    df_transcripts.to_csv(transcripts_csv, index=False, encoding='utf-8')

videos_json = OUTPUT_DIR / f'youtube_videos_nursing_{timestamp}.json'
with open(videos_json, 'w', encoding='utf-8') as f:
    json.dump(all_videos, f, ensure_ascii=False, indent=2, default=str)

log.info('Saved:')
log.info(f'  {comments_json}     ({len(all_comments)} comments)')
log.info(f'  {transcripts_json}  ({len(all_transcripts)} transcripts)')
log.info(f'  {videos_json}       ({len(all_videos)} videos)')

## Next steps

1. **Eyeball the CSVs** in `data/raw/` — verify content is actually nursing-related.
2. **Preprocessing (Week 3)** — apply the cleaning function from the old notebook to both `df_comments['text']` and `df_transcripts['text']`. Keep the `source` column intact through preprocessing. When you run sentiment analysis, look at trends for comments and transcripts separately first, then combined.
3. **Reddit collection** — Week 2 also calls for Reddit data. Volume and historical depth on r/nursing will dwarf YouTube. Mirror this notebook's structure.
4. **Expand to other occupations** — add entries to `OCCUPATIONS` and re-run. Output filenames are date-stamped so old runs aren't overwritten.

## Troubleshooting

- **All transcripts failed with `IpBlocked` / `RequestBlocked`:** YouTube has blocked your network. Try a different network (home → uni, or vice versa). Cloud notebook environments (Colab, Kaggle, Codespaces) are almost always blocked.
- **Some transcripts failed with `NoTranscriptFound`:** the channel didn't enable auto-captions and didn't upload manual ones. Nothing you can do — collect more videos to compensate.
- **Quota exceeded:** YouTube Data API gives 10,000 units/day. Search costs 100 per call, commentThreads is 1 per call. Transcript fetching is FREE (doesn't use the official API). Wait until UTC midnight or request a quota increase via Google Cloud Console.
- **Empty CSV files:** check the logs above the save cell — every failure is logged with a reason, unlike the bare `except:` in the old notebook.